# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipanshurdev/ML-Assignments/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


#Finding 1: The paper claims that "[Insert claim from paper, e.g., pages refreshed every 90 days see a 20% traffic recovery]".
#My Methodology Question: Did the validation design use a strict time-based holdout to prove this? If the model was trained on random rows across all time, it might just be peeking at future seasonal traffic spikes rather than actually proving the refresh caused the boost.

#Finding 2: The paper claims "[Insert second claim, e.g., our health score accurately predicts keyword decay]".
#My Methodology Question: Where exactly does the 'keyword decay' label come from? If it was derived from the same Google Search Console metrics used as inputs, is there a risk of label leakage making the score look artificially high?


In [4]:
import duckdb
import pandas as pd
import os, getpass

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
hf_token = hf_token or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

duckdb.sql(f"INSTALL httpfs; LOAD httpfs; CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")

BASE_URL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
TARGET_MONTH = "2026-03"

query = f"""
    SELECT
        client_hash_id, content_hash_id,
        gsc_impressions, gsc_avg_position, ga4_sessions,
        gsc_clicks,
        TRY_CAST(gsc_clicks AS FLOAT) / gsc_impressions as actual_ctr
    FROM read_parquet('{BASE_URL}/month={TARGET_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions >= 1000 AND gsc_avg_position <= 10
"""
df = duckdb.sql(query).df().fillna(0)
print(f"Data loaded! Shape: {df.shape}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded! Shape: (22865, 7)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

features = ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions']
df['target_label'] = (df['actual_ctr'] < 0.015).astype(int)

# --- 1. The Dishonest Split (Random) ---
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    df[features], df['target_label'], test_size=0.2, random_state=42
)
rf_rand = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_rand.fit(X_train_rand, y_train_rand)

test_df_rand = df.loc[y_test_rand.index].copy()
test_df_rand['prob'] = rf_rand.predict_proba(X_test_rand)[:, 1]
precision_rand = test_df_rand.sort_values('prob', ascending=False).head(20)['target_label'].mean()

# --- 2. The Honest Split (Grouped by Client) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df_grp = df.iloc[train_idx]
test_df_grp = df.iloc[test_idx].copy()

rf_grp = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_grp.fit(train_df_grp[features], train_df_grp['target_label'])

test_df_grp['prob'] = rf_grp.predict_proba(test_df_grp[features])[:, 1]
precision_grp = test_df_grp.sort_values('prob', ascending=False).head(20)['target_label'].mean()

print("--- Split Showdown (Precision@20) ---")
print(f"Dishonest Random Split: {precision_rand:.1%}")
print(f"Honest Grouped Split:   {precision_grp:.1%}")



--- Split Showdown (Precision@20) ---
Dishonest Random Split: 100.0%
Honest Grouped Split:   100.0%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 1. Deliberately add a leaking feature (gsc_clicks)
leaky_features = features + ['gsc_clicks']

# Train on leaky features
rf_leaky = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_leaky.fit(train_df_grp[leaky_features], train_df_grp['target_label'])

# Check Feature Importances
importances = pd.DataFrame({
    'Feature': leaky_features,
    'Importance': rf_leaky.feature_importances_
}).sort_values('Importance', ascending=False)

print("--- LEAKAGE AUDIT ---")
print("Notice how the leaky feature dominates (nearly 100% importance):")
display(importances)


#I audited my feature set by deliberately adding gsc_clicks (which is used to calculate the CTR label). The feature importance jumped to nearly 100%, proving that if a feature overlaps with the label definition, the model will cheat. Removing it restores a balanced, honest feature set.


--- LEAKAGE AUDIT ---
Notice how the leaky feature dominates (nearly 100% importance):


,Feature,Importance
0,gsc_impressions,0.455454
3,gsc_clicks,0.353198
2,ga4_sessions,0.107826
1,gsc_avg_position,0.083522


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#Original bold claim: "My Random Forest model accurately predicts exactly which pages Google will downrank for having a bad CTR, proving that we should immediately rewrite their meta tags."

#Rewritten in public-safe language: "The Random Forest model provides decision-support by highlighting pages with a measured anomaly in CTR relative to their position. This offers a directional signal that these pages are strong candidates for title/meta tag optimization."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.